# Prophet Baseline

Đánh giá Prophet trên toàn bộ 100 series (Store × Product), theo cùng convention với `run_naive.py` và `run_lstm2.py`.

In [37]:
#  import os

# print(os.listdir("/kaggle/input/datasets/thaonngyn/retail-data/sales_data.csv"))

In [38]:
import numpy as np
import pandas as pd
import warnings
import sys, os
from prophet import Prophet

import os
import warnings
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
warnings.filterwarnings('ignore')
sys.path.append('..')

2026-03-18 00:17:20.698681: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773793041.036364      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773793041.131581      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773793041.901851      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773793041.901910      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773793041.901913      55 computation_placer.cc:177] computation placer alr

In [39]:
DATA_PATH  = '/kaggle/input/datasets/thaonngyn/retail-data/sales_data.csv'
RESULT_DIR = '/kaggle/working/'
TARGET     = 'Units Sold'
TRAIN_END  = '2023-06-30'
VAL_END    = '2023-10-31'
HORIZONS   = [7, 14, 28]
LOOKBACK   = 30
LAG        = 7

In [40]:
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store ID', 'Product ID', 'Date'])

series_dict = {
    f"{store}_{product}": grp.set_index('Date')[TARGET]
    for (store, product), grp in df.groupby(['Store ID', 'Product ID'])
}
series_ids = sorted(series_dict.keys())
print(f'Total series: {len(series_ids)}')

Total series: 100


In [41]:
def prophet_fn(train_dates, train_values, horizon):
    df_fit = pd.DataFrame({'ds': train_dates, 'y': train_values.astype(float)})
    m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=False)
    m.fit(df_fit)
    future = m.make_future_dataframe(periods=horizon, freq='D', include_history=False)
    fc = m.predict(future)['yhat'].values
    return np.clip(fc, 0, None)


def rolling_eval(series, split_end, horizon, lookback):
    eval_start = pd.Timestamp(split_end) + pd.Timedelta(days=1)
    eval_end   = series.index.max()
    all_fc, all_ac, last_train = [], [], None
    t = eval_start
    while t + pd.Timedelta(days=horizon - 1) <= eval_end:
        window = series[t - pd.Timedelta(days=lookback): t - pd.Timedelta(days=1)]
        actual = series[t: t + pd.Timedelta(days=horizon - 1)].values.astype(float)
        if len(window) < 2 or len(actual) < horizon:
            t += pd.Timedelta(days=horizon); continue
        all_fc.append(prophet_fn(window.index, window.values, horizon))
        all_ac.append(actual)
        last_train = window.values.astype(float)
        t += pd.Timedelta(days=horizon)
    if not all_fc:
        return {k: np.nan for k in ['smape', 'mase', 'rmse', 'rmsle']}
    fc_arr = np.array(all_fc)
    ac_arr = np.array(all_ac)
    lag    = min(LAG, len(last_train) - 1)
    denom  = np.mean(np.abs(last_train[lag:] - last_train[:-lag])) or 1.0
    return {
        'smape': (2 * np.abs(fc_arr - ac_arr) / (np.abs(fc_arr) + np.abs(ac_arr) + 1e-8)).mean() * 100,
        'mase' : np.mean(np.abs(fc_arr - ac_arr)) / denom,
        'rmse' : np.sqrt(np.mean((fc_arr - ac_arr) ** 2)),
        'rmsle': np.sqrt(np.mean((np.log1p(fc_arr) - np.log1p(ac_arr)) ** 2)),
    }

In [42]:
os.makedirs(RESULT_DIR, exist_ok=True)
results = []

for h in HORIZONS:
    scores = {k: [] for k in ['smape', 'mase', 'rmse', 'rmsle']}
    for i, sid in enumerate(series_ids):
        r = rolling_eval(series_dict[sid], VAL_END, h, LOOKBACK)
        for k in scores:
            scores[k].append(r[k])
        if (i + 1) % 20 == 0:
            print(f'  H={h} | {i+1}/{len(series_ids)} done')
    row = {
        'model': 'Prophet', 'dataset': 'retail_inventory_daily', 'target': TARGET,
        'horizon': h, 'lookback': LOOKBACK,
        'mean_smape':   round(float(np.nanmean(scores['smape'])), 4),
        'median_smape': round(float(np.nanmedian(scores['smape'])), 4),
        'mean_mase':    round(float(np.nanmean(scores['mase'])), 4),
        'median_mase':  round(float(np.nanmedian(scores['mase'])), 4),
        'mean_rmse':    round(float(np.nanmean(scores['rmse'])), 4),
        'median_rmse':  round(float(np.nanmedian(scores['rmse'])), 4),
        'mean_rmsle':   round(float(np.nanmean(scores['rmsle'])), 4),
        'median_rmsle': round(float(np.nanmedian(scores['rmsle'])), 4),
    }
    results.append(row)
    print(f"H={h:2d} | sMAPE={row['mean_smape']:.2f}% MASE={row['mean_mase']:.4f} RMSE={row['mean_rmse']:.2f} RMSLE={row['mean_rmsle']:.4f}")

pd.DataFrame(results).to_csv(f'{RESULT_DIR}/prophet_daily_summary.csv', index=False)
print(f'\nSaved to {RESULT_DIR}/prophet_daily_summary.csv')

00:17:53 - cmdstanpy - INFO - Chain [1] start processing
00:17:53 - cmdstanpy - INFO - Chain [1] done processing
00:17:53 - cmdstanpy - INFO - Chain [1] start processing
00:17:53 - cmdstanpy - INFO - Chain [1] done processing
00:17:53 - cmdstanpy - INFO - Chain [1] start processing
00:17:53 - cmdstanpy - INFO - Chain [1] done processing
00:17:53 - cmdstanpy - INFO - Chain [1] start processing
00:17:53 - cmdstanpy - INFO - Chain [1] done processing
00:17:53 - cmdstanpy - INFO - Chain [1] start processing
00:17:53 - cmdstanpy - INFO - Chain [1] done processing
00:17:53 - cmdstanpy - INFO - Chain [1] start processing
00:17:53 - cmdstanpy - INFO - Chain [1] done processing
00:17:53 - cmdstanpy - INFO - Chain [1] start processing
00:17:54 - cmdstanpy - INFO - Chain [1] done processing
00:17:54 - cmdstanpy - INFO - Chain [1] start processing
00:17:54 - cmdstanpy - INFO - Chain [1] done processing
00:17:54 - cmdstanpy - INFO - Chain [1] start processing
00:17:54 - cmdstanpy - INFO - Chain [1]

  H=7 | 20/100 done


00:18:29 - cmdstanpy - INFO - Chain [1] done processing
00:18:29 - cmdstanpy - INFO - Chain [1] start processing
00:18:29 - cmdstanpy - INFO - Chain [1] done processing
00:18:29 - cmdstanpy - INFO - Chain [1] start processing
00:18:29 - cmdstanpy - INFO - Chain [1] done processing
00:18:29 - cmdstanpy - INFO - Chain [1] start processing
00:18:29 - cmdstanpy - INFO - Chain [1] done processing
00:18:29 - cmdstanpy - INFO - Chain [1] start processing
00:18:29 - cmdstanpy - INFO - Chain [1] done processing
00:18:29 - cmdstanpy - INFO - Chain [1] start processing
00:18:29 - cmdstanpy - INFO - Chain [1] done processing
00:18:29 - cmdstanpy - INFO - Chain [1] start processing
00:18:29 - cmdstanpy - INFO - Chain [1] done processing
00:18:30 - cmdstanpy - INFO - Chain [1] start processing
00:18:30 - cmdstanpy - INFO - Chain [1] done processing
00:18:30 - cmdstanpy - INFO - Chain [1] start processing
00:18:30 - cmdstanpy - INFO - Chain [1] done processing
00:18:30 - cmdstanpy - INFO - Chain [1] 

  H=7 | 40/100 done


00:19:04 - cmdstanpy - INFO - Chain [1] done processing
00:19:04 - cmdstanpy - INFO - Chain [1] start processing
00:19:04 - cmdstanpy - INFO - Chain [1] done processing
00:19:04 - cmdstanpy - INFO - Chain [1] start processing
00:19:05 - cmdstanpy - INFO - Chain [1] done processing
00:19:05 - cmdstanpy - INFO - Chain [1] start processing
00:19:05 - cmdstanpy - INFO - Chain [1] done processing
00:19:05 - cmdstanpy - INFO - Chain [1] start processing
00:19:05 - cmdstanpy - INFO - Chain [1] done processing
00:19:05 - cmdstanpy - INFO - Chain [1] start processing
00:19:05 - cmdstanpy - INFO - Chain [1] done processing
00:19:05 - cmdstanpy - INFO - Chain [1] start processing
00:19:05 - cmdstanpy - INFO - Chain [1] done processing
00:19:05 - cmdstanpy - INFO - Chain [1] start processing
00:19:05 - cmdstanpy - INFO - Chain [1] done processing
00:19:05 - cmdstanpy - INFO - Chain [1] start processing
00:19:05 - cmdstanpy - INFO - Chain [1] done processing
00:19:05 - cmdstanpy - INFO - Chain [1] 

...

  H=28 | 60/100 done


00:22:43 - cmdstanpy - INFO - Chain [1] done processing
00:22:43 - cmdstanpy - INFO - Chain [1] start processing
00:22:44 - cmdstanpy - INFO - Chain [1] done processing
00:22:44 - cmdstanpy - INFO - Chain [1] start processing
00:22:44 - cmdstanpy - INFO - Chain [1] done processing
00:22:44 - cmdstanpy - INFO - Chain [1] start processing
00:22:44 - cmdstanpy - INFO - Chain [1] done processing
00:22:44 - cmdstanpy - INFO - Chain [1] start processing
00:22:44 - cmdstanpy - INFO - Chain [1] done processing
00:22:44 - cmdstanpy - INFO - Chain [1] start processing
00:22:44 - cmdstanpy - INFO - Chain [1] done processing
00:22:44 - cmdstanpy - INFO - Chain [1] start processing
00:22:44 - cmdstanpy - INFO - Chain [1] done processing
00:22:44 - cmdstanpy - INFO - Chain [1] start processing
00:22:44 - cmdstanpy - INFO - Chain [1] done processing
00:22:44 - cmdstanpy - INFO - Chain [1] start processing
00:22:45 - cmdstanpy - INFO - Chain [1] done processing
00:22:45 - cmdstanpy - INFO - Chain [1] 

  H=28 | 80/100 done


00:22:52 - cmdstanpy - INFO - Chain [1] done processing
00:22:52 - cmdstanpy - INFO - Chain [1] start processing
00:22:52 - cmdstanpy - INFO - Chain [1] done processing
00:22:52 - cmdstanpy - INFO - Chain [1] start processing
00:22:52 - cmdstanpy - INFO - Chain [1] done processing
00:22:52 - cmdstanpy - INFO - Chain [1] start processing
00:22:52 - cmdstanpy - INFO - Chain [1] done processing
00:22:52 - cmdstanpy - INFO - Chain [1] start processing
00:22:53 - cmdstanpy - INFO - Chain [1] done processing
00:22:53 - cmdstanpy - INFO - Chain [1] start processing
00:22:53 - cmdstanpy - INFO - Chain [1] done processing
00:22:53 - cmdstanpy - INFO - Chain [1] start processing
00:22:53 - cmdstanpy - INFO - Chain [1] done processing
00:22:53 - cmdstanpy - INFO - Chain [1] start processing
00:22:53 - cmdstanpy - INFO - Chain [1] done processing
00:22:53 - cmdstanpy - INFO - Chain [1] start processing
00:22:53 - cmdstanpy - INFO - Chain [1] done processing
00:22:53 - cmdstanpy - INFO - Chain [1] 

  H=28 | 100/100 done
H=28 | sMAPE=49.52% MASE=0.9726 RMSE=51.13 RMSLE=0.7600

Saved to /kaggle/working//prophet_daily_summary.csv


In [43]:
pd.DataFrame(results)

,model,dataset,target,horizon,lookback,mean_smape,median_smape,mean_mase,median_mase,mean_rmse,median_rmse,mean_rmsle,median_rmsle
0,Prophet,retail_inventory_daily,Units Sold,7,30,46.3844,44.8046,0.9666,0.9449,47.2201,47.6378,0.7572,0.7368
1,Prophet,retail_inventory_daily,Units Sold,14,30,54.6941,52.5017,1.0382,1.0357,52.4637,51.8216,0.9626,0.8795
2,Prophet,retail_inventory_daily,Units Sold,28,30,49.5213,47.1424,0.9726,0.9124,51.1282,49.3147,0.7600,0.6990
